# Module 4 — Xuất báo cáo Excel SCADA

Notebook đọc dữ liệu sạch `Data/output/du_lieu_sach.csv` và xuất file báo cáo Excel `Report/BaoCao_SCADA.xlsx` gồm 3 sheet:

1. **Chi_so_chung** — các chỉ số tổng quan
2. **Tong_hop_thang** — tổng hợp theo tháng
3. **Diem_su_co** — danh sách điểm nghi sự cố

Điểm nghi sự cố = gió > 5 m/s **và** công suất < 50% công suất lý thuyết. Mỗi điểm tương ứng 10 phút.

## Bước 1 — Đọc dữ liệu và xác định điểm nghi sự cố

In [1]:
import os
import pandas as pd

CSV_INPUT = "../Data/output/du_lieu_sach.csv"
XLSX_OUTPUT = "../Report/BaoCao_SCADA.xlsx"
PHUT_MOI_DIEM = 10  # mỗi bản ghi cách nhau 10 phút

# Đọc dữ liệu sạch; thoi_gian làm chỉ mục thời gian
df = pd.read_csv(CSV_INPUT, index_col="thoi_gian", parse_dates=True)

# Cờ điểm nghi sự cố: gió mạnh nhưng công suất thấp bất thường
df["nghi_su_co"] = (df["toc_do_gio"] > 5) & (df["cong_suat_kw"] < 0.5 * df["cong_suat_ly_thuyet"])

print(f"Tổng số bản ghi: {len(df)}")
print(f"Số điểm nghi sự cố: {int(df['nghi_su_co'].sum())}")

Tổng số bản ghi: 48835
Số điểm nghi sự cố: 2219


## Bước 2 — Sheet "Chi_so_chung": các chỉ số tổng quan

In [2]:
so_diem_su_co = int(df["nghi_su_co"].sum())
# Tổng thời gian sự cố quy ra giờ: mỗi điểm 10 phút
tong_gio_su_co = so_diem_su_co * PHUT_MOI_DIEM / 60

chi_so_chung = pd.DataFrame({
    "Chỉ số": [
        "Tổng số bản ghi",
        "Công suất trung bình (kW)",
        "Tốc độ gió trung bình (m/s)",
        "Số điểm nghi sự cố",
        "Tổng thời gian sự cố (giờ)",
    ],
    "Giá trị": [
        len(df),
        round(df["cong_suat_kw"].mean(), 2),
        round(df["toc_do_gio"].mean(), 2),
        so_diem_su_co,
        round(tong_gio_su_co, 2),
    ],
})

print("Sheet Chi_so_chung:")
chi_so_chung

Sheet Chi_so_chung:


,Chỉ số,Giá trị
0,Tổng số bản ghi,48835.00
1,Công suất trung bình (kW),1225.70
2,Tốc độ gió trung bình (m/s),7.22
3,Số điểm nghi sự cố,2219.00
4,Tổng thời gian sự cố (giờ),369.83


## Bước 3 — Sheet "Tong_hop_thang": tổng hợp theo tháng

In [3]:
# Gộp theo tháng (month-end)
gom_thang = df.resample("ME")
tong_hop_thang = pd.DataFrame({
    # Sản lượng (MWh) = tổng công suất(kW) * 10 phút / 60 / 1000
    "Tong_san_luong_MWh": (gom_thang["cong_suat_kw"].sum() * PHUT_MOI_DIEM / 60 / 1000).round(2),
    "Cong_suat_TB_kW": gom_thang["cong_suat_kw"].mean().round(2),
    "Toc_do_gio_TB_ms": gom_thang["toc_do_gio"].mean().round(2),
    "So_diem_su_co": gom_thang["nghi_su_co"].sum().astype(int),
})
# Nhãn tháng dạng MM/YYYY cho dễ đọc
tong_hop_thang.index = tong_hop_thang.index.strftime("%m/%Y")
tong_hop_thang.index.name = "Thang"

print("Sheet Tong_hop_thang:")
tong_hop_thang

Sheet Tong_hop_thang:


,Tong_san_luong_MWh,Cong_suat_TB_kW,Toc_do_gio_TB_ms,So_diem_su_co
Thang,,,,
01/2018,808.74,1288.14,8.45,808
02/2018,773.60,1274.82,7.49,353
03/2018,1166.83,1752.43,8.88,89
04/2018,537.27,764.43,5.61,116
05/2018,620.50,836.82,5.86,46
06/2018,683.05,973.01,6.27,12
07/2018,354.92,477.05,4.95,13
08/2018,1421.12,1953.88,9.25,49
09/2018,932.53,1410.43,7.52,13


## Bước 4 — Sheet "Diem_su_co": danh sách điểm nghi sự cố

In [4]:
# Lọc các điểm nghi sự cố, giữ các cột cần thiết
diem_su_co = (
    df.loc[df["nghi_su_co"], ["toc_do_gio", "cong_suat_kw", "cong_suat_ly_thuyet"]]
    .rename(columns={
        "toc_do_gio": "Toc_do_gio_ms",
        "cong_suat_kw": "Cong_suat_thuc_kW",
        "cong_suat_ly_thuyet": "Cong_suat_ly_thuyet_kWh",
    })
    .round(2)
)
diem_su_co.index.name = "Thoi_gian"

print(f"Sheet Diem_su_co: {len(diem_su_co)} dòng")
diem_su_co.head()

Sheet Diem_su_co: 2219 dòng


,Toc_do_gio_ms,Cong_suat_thuc_kW,Cong_suat_ly_thuyet_kWh
Thoi_gian,,,
2018-01-04 15:00:00,5.16,0.00,376.31
2018-01-04 15:10:00,5.16,0.00,376.68
2018-01-04 15:20:00,5.41,0.00,444.87
2018-01-04 15:30:00,6.78,0.00,919.37
2018-01-11 09:10:00,10.88,558.43,3228.38


## Bước 5 — Ghi ra file Excel (3 sheet) và kiểm chứng số dòng

In [5]:
# Tạo folder Report nếu chưa có
os.makedirs(os.path.dirname(XLSX_OUTPUT), exist_ok=True)

# Ghi 3 sheet bằng pandas ExcelWriter (engine openpyxl)
with pd.ExcelWriter(XLSX_OUTPUT, engine="openpyxl") as writer:
    chi_so_chung.to_excel(writer, sheet_name="Chi_so_chung", index=False)
    tong_hop_thang.to_excel(writer, sheet_name="Tong_hop_thang", index=True)
    diem_su_co.to_excel(writer, sheet_name="Diem_su_co", index=True)

print(f"Đã lưu báo cáo: {XLSX_OUTPUT}\n")

# Kiểm chứng: in số dòng dữ liệu của mỗi sheet
print("Số dòng dữ liệu mỗi sheet (chưa tính dòng tiêu đề):")
print(f"  - Chi_so_chung  : {len(chi_so_chung)} dòng")
print(f"  - Tong_hop_thang: {len(tong_hop_thang)} dòng")
print(f"  - Diem_su_co    : {len(diem_su_co)} dòng")

Đã lưu báo cáo: ../Report/BaoCao_SCADA.xlsx

Số dòng dữ liệu mỗi sheet (chưa tính dòng tiêu đề):
  - Chi_so_chung  : 5 dòng
  - Tong_hop_thang: 12 dòng
  - Diem_su_co    : 2219 dòng
